In [ ]:
from datetime import datetime
import uuid
import requests
import os
import pandas as pd

SOURCE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2024-01.parquet"
)

BRONZE_PATH = (
    "/Volumes/dbw_stratum_dev/bronze/"
    "nyc_taxi/yellow/2024/01/"
)

LOCAL_PATH = "/tmp/yellow_tripdata_2024_01.parquet"

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.now()

print(f"Run ID: {RUN_ID}")
print(f"Started: {STARTED_AT}")
print(f"Source: {SOURCE_URL}")
print(f"Target: {BRONZE_PATH}")


spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_stratum_dev.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_stratum_dev.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_stratum_dev.gold")
spark.sql(
    "CREATE VOLUME IF NOT EXISTS "
    "dbw_stratum_dev.bronze.nyc_taxi"
)

print("Catalog structure verified")


print(f"\nDownloading source file...")
print("Takes 2-3 minutes...")

response = requests.get(SOURCE_URL, stream=True)
response.raise_for_status()

total_bytes = 0
with open(LOCAL_PATH, "wb") as f:
    for chunk in response.iter_content(
        chunk_size=8192
    ):
        f.write(chunk)
        total_bytes += len(chunk)

size_mb = total_bytes / (1024 * 1024)
print(f"Downloaded: {size_mb:.1f} MB")

print("\nReading with pandas...")

pdf = pd.read_parquet(LOCAL_PATH)

rows_read = len(pdf)
print(f"Rows read: {rows_read:,}")
print(f"Columns: {len(pdf.columns)}")


print("\nConverting to Spark DataFrame...")
df_raw = spark.createDataFrame(pdf)

print(f"\nWriting to bronze at {BRONZE_PATH}...")

(df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "false")
    .save(BRONZE_PATH))

print("Write complete")


df_verify = spark.read.format("delta").load(
    BRONZE_PATH
)

rows_written = df_verify.count()

assert rows_written == rows_read, (
    f"Row count mismatch. "
    f"Read {rows_read:,} but wrote {rows_written:,}."
)

completed_at = datetime.now()
duration = (completed_at - STARTED_AT).total_seconds()

print(f"\nVerification:")
print(f"Rows in bronze: {rows_written:,}")
print(f"Duration: {duration:.1f}s")
print(f"Status: SUCCESS")

display(df_verify.limit(5))

df_verify.printSchema()

display(
    spark.sql(f"""
        DESCRIBE HISTORY
        delta.`{BRONZE_PATH}`
    """)
)

os.remove(LOCAL_PATH)
print(f"\nTemp file removed")

print(f"\nRun complete:")
print(f"  Run ID: {RUN_ID}")
print(f"  Rows read: {rows_read:,}")
print(f"  Rows written: {rows_written:,}")
print(f"  Duration: {duration:.1f}s")
print(f"  Status: SUCCESS")
print(f"\nNext step: run 02_register_bronze_tables")